# Goal Inference

This example shows how to evaluate a a `genlm.control` model on the Planetarium Blocksworld goal-inference task.

* **Task**: Given a natural-language description, generate the PDDL goal S-expression that completes (:goal (and …)) for a provided problem.

* **Data**: [Planetarium] (Zuo et al., 2024) Blocksworld subset, filtered to goals of the form (:goal (and …)) and small instances (e.g., < 10 objects).

## Setup

First, install the dependencies for this domain. In the root directory, run:    

```bash
pip install -e .[goal_inference]
```

## Install Fast Downward and VAL

The benchmarks requires the Fast-Downward planner, and the VAL plan validator.

To install them on Linux, follow the instructions on `https://github.com/BatsResearch/planetarium`:
```
apptainer pull fast-downward.sif docker://aibasel/downward:latest
mkdir tmp
curl -o tmp/VAL.zip https://dev.azure.com/schlumberger/4e6bcb11-cd68-40fe-98a2-e3777bfec0a6/_apis/build/builds/77/artifacts?artifactName=linux64\&api-version=7.1\&%24format=zip
unzip tmp/VAL.zip -d tmp/
tar -xzvf tmp/linux64/*.tar.gz -C tmp/ --strip-components=1
```

For other platforms follow the instructions under `https://github.com/aibasel/downward/blob/main/BUILD.md` 


Make sure to add fast-downward.sif and VAL to your PATH or make aliases.

In [1]:
import os
from pathlib import Path

os.environ["PATH"] = str(Path("./../../tmp/bin")) + os.pathsep + os.environ["PATH"]

## Verify the commands are working

In [2]:
!Validate -h | head -n 5
!../../../fast-downward.sif -h | head -n 5

VAL: The PDDL+ plan validation tool
Version 4: Validates continuous effects, events and processes.

Authors: Derek Long, Richard Howey, Stephen Cresswell and Maria Fox
https:://github/KCL-Planning/VAL
usage: fast-downward.py [-h] [-v] [--show-aliases] [--run-all] [--translate]
                        [--search]
                        [--translate-time-limit TRANSLATE_TIME_LIMIT]
                        [--translate-memory-limit TRANSLATE_MEMORY_LIMIT]
                        [--search-time-limit SEARCH_TIME_LIMIT]


## Usage 

### Initialize the dataset and evaluator

In [4]:
from genlm.eval.domains.goal_inference import (
    GoalInferenceDataset,
    GoalInferenceEvaluator,
)

/home/samuelkiegeland/micromamba/envs/py311-goal-inference/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 10-08 10:07:04 [__init__.py:235] Automatically detected platform cuda.


In [5]:
dataset = GoalInferenceDataset.from_hf_planetarium(
    split="train", subset="default", max_objects=2, domains=["blocksworld"]
)

evaluator = GoalInferenceEvaluator()

### Define a model adaptor

A model adaptor is an async callable that takes a `PatternMatchingInstance` and returns a `ModelOutput`. Here we'll use a `genlm.control.PromptedLLM` constrained to PDDL goals (via the `GoalInferenceVALPotential` potential).

In [6]:
from genlm.control import PromptedLLM, AWRS
from genlm.eval import ModelOutput, ModelResponse
from genlm.eval.domains.goal_inference import (
    goal_default_prompt_formatter,
    GoalInferenceVALPotential,
)

# Read domain pddl.
with open("../../../assets/goal_inference/pddl_domains/blocksworld.pddl") as f:
    domain_text = f.read()

# Load an LLM - For better performance, choose a larger model like "meta-llama/Meta-Llama-3-8B"
LLM = PromptedLLM.from_name("meta-llama/Meta-Llama-3-8B")


async def model(instance, output_dir, replicate):
    # Set the prompt for the LLM.
    LLM.prompt_ids = goal_default_prompt_formatter(
        LLM.model.tokenizer, instance, use_chat_format=False
    )

    # Construct goal validation potential.
    potential = GoalInferenceVALPotential(
        domain_pddl_text=domain_text,
        problem_pddl_text=instance.problem_text,
        fast_downward_cmd="../../../fast-downward.sif",
        val_cmd="Validate",
        cache_root=".cache",
        verbosity=0,
    ).coerce(LLM, f=b"".join)

    # Define an adaptive weighted rejection sampler to sample tokens from the constrained model.
    sampler = AWRS(LLM, potential)

    # Run SMC to sample sequences from the constrained model.
    sequences = await sampler.smc(
        n_particles=10,
        ess_threshold=0.9,
        max_tokens=100,
    )

    # Return the sampled sequences and their probabilities as a ModelOutput.
    return ModelOutput(
        responses=[
            ModelResponse(response=sequence, weight=prob)
            for sequence, prob in sequences.decoded_posterior.items()
        ],
    )

`torch_dtype` is deprecated! Use `dtype` instead!


INFO 10-08 10:07:27 [config.py:1604] Using max model len 8192
WARNING 10-08 10:07:27 [arg_utils.py:1690] --disable-async-output-proc is not supported by the V1 Engine. Falling back to V0. We recommend to remove --disable-async-output-proc from your config in favor of the V1 Engine.


2025-10-08 10:07:28,012	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 10-08 10:07:28 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='meta-llama/Meta-Llama-3-8B', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_backend=''), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None), seed=0, served_model_name=meta-llama/Meta-Llama-3-8B, num_scheduler_steps=1, multi_step_stream_outputs=True, enable_

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


...

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:24<01:12, 24.27s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:48<00:48, 24.24s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:54<00:15, 15.77s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [01:18<00:00, 18.99s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [01:18<00:00, 19.53s/it]


INFO 10-08 10:08:50 [default_loader.py:262] Loading weights took 78.21 seconds


INFO 10-08 10:08:50 [model_runner.py:1115] Model loading took 14.9596 GiB and 78.852735 seconds
INFO 10-08 10:08:53 [worker.py:295] Memory profiling takes 2.75 seconds
INFO 10-08 10:08:53 [worker.py:295] the current vLLM instance can use total_gpu_memory (22.05GiB) x gpu_memory_utilization (0.90) = 19.84GiB
INFO 10-08 10:08:53 [worker.py:295] model weights take 14.96GiB; non_torch_memory takes 0.04GiB; PyTorch activation peak memory takes 1.23GiB; the rest of the memory reserved for KV Cache is 3.60GiB.
INFO 10-08 10:08:54 [executor_base.py:113] # cuda blocks: 1844, # CPU blocks: 2048
INFO 10-08 10:08:54 [executor_base.py:118] Maximum concurrency for 8192 tokens per request: 3.60x
INFO 10-08 10:08:56 [model_runner.py:1385] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decr

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:29<00:00,  1.20it/s]

INFO 10-08 10:09:25 [model_runner.py:1537] Graph capturing finished in 29 secs, took 0.26 GiB
INFO 10-08 10:09:25 [llm_engine.py:424] init engine (profile, create kv cache, warmup model) took 35.11 seconds


### Run the evaluation

In [ ]:
from genlm.eval import run_evaluation

results = await run_evaluation(
    dataset=dataset,
    model=model,
    evaluator=evaluator,
    max_instances=3,
    n_replicates=1,
    verbosity=3,
    #output_dir="goal_inference_results", #optionally save the results to a directory
)

Here 1
Here 2
Engery 1:  
Engery 1:  
Engery 1:  
Engery 1:  
Engery 1:  
Engery 1:  
Engery 1:  
Engery 1:  
Engery 1:  
Engery 1:  
Engery 1:   (
Engery 1:   (
Engery 1:   (
Engery 1:   (
Engery 1:   (
Engery 1:   (
Engery 1:   (
Engery 1:   (
Engery 1:   (
Engery 1:   (
Engery 1:   (on
Engery 1:   (stack
Engery 1:   (pose
Engery 1:   (stack
Engery 1:   (stack
Engery 1:   (block
Engery 1:   (h
Engery 1:   (stack
Engery 1:   (stack
Engery 1:   (on
Engery 1:   (on
Engery 1:   (stack
Engery 1:   (holding
Engery 1:   (on
Engery 1:   (stack
Engery 1:   (Single
Engery 1:   (holding
Engery 1:   (on
Engery 1:   (on
Engery 1:   (stack
Engery 1:   (on-top
Engery 1:   (on-table
Engery 1:   (pose-after
Engery 1:   (pose (
Engery 1:   (stack b
Engery 1:   (stacked
Engery 1:   (h1
Engery 1:   (h.on
Engery 1:   (stack b
Engery 1:   (stack)

Engery before
done command, val with cmd:  Validate .cache/pddl_tasks/226721c786006dfcd49785f6f8f665fbefabf9ee9714d45cda1404e6e178811d.domain.pddl /tmp/val_goal

## References

Max Zuo, Francisco Piedrahita Velez, Xiaochen Li, Michael L. Littman, and Stephen H. Bach. Planetarium: a rigorous benchmark for translating text to structured planning languages. arXiv preprint arXiv:2407.03321, 2024. URL https://arxiv.org/abs/2407.03321